# 03 Reconciliation gate
Before I derive anything new I rebuild the published annual figures from my own monthly tables, and I publish the residual rather than tidy it away.

I report **two definitions** because the first one I tried did not reconcile. The report-table text says planned attendances are excluded, but the published numbers only reconcile once booked-appointment attendances are included. I keep both rows in the output so the difference stays visible.

In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import numpy as np
import common as c
pd.options.display.float_format = '{:,.4f}'.format
pd.options.display.width = 200


In [2]:
m = pd.read_parquet(c.SILVER / 'msitae_monthly_provider.parquet')
t1 = pd.read_parquet(c.SILVER / 'msitae_table1_attendances.parquet').set_index('year')
t3 = pd.read_parquet(c.SILVER / 'msitae_table3_four_hour.parquet').set_index('year')
t6 = pd.read_parquet(c.SILVER / 'msitae_table6_type1_provider.parquet')
t11 = pd.read_parquet(c.SILVER / 'msitae_table11_dept_type.parquet').set_index('dept_type')

def rebuild(include_booked):
    att = m[c.ATT_COLS].sum()
    over = m[c.OVER4_COLS].sum()
    if include_booked:
        att = att + m[c.BOOKED_COLS].sum().set_axis(c.ATT_COLS)
        over = over + m[c.OVER4_BOOKED_COLS].sum().set_axis(c.OVER4_COLS)
    return att, over

rows = []
for label, inc in [('non-booked only', False), ('non-booked + booked', True)]:
    att, over = rebuild(inc)
    pub_type = {'att_t1': t11.msitae.iloc[0], 'att_t2': t11.msitae.iloc[1], 'att_other': t11.msitae.iloc[2]}
    rows += [('attendances, all types', label, att.sum(), t1.loc['2025-26', 'attendances']),
             ('four-hour %', label, 100 * (1 - over.sum() / att.sum()), 100 * t3.loc['2025-26', 'pct_le4'])]
    rows += [(f'attendances, {k}', label, att[k], v) for k, v in pub_type.items()]
gate = pd.DataFrame(rows, columns=['metric', 'definition', 'rebuilt', 'published'])
gate['residual'] = gate.rebuilt - gate.published
gate['residual_pct'] = 100 * gate.residual / gate.published
gate

,metric,definition,rebuilt,published,residual,residual_pct
0,"attendances, all types",non-booked only,"26,969,593.0000","27,976,025.0000","-1,006,432.0000",-3.5975
1,four-hour %,non-booked only,74.3660,74.9288,-0.5627,-0.7510
2,"attendances, att_t1",non-booked only,"16,744,864.0000","17,121,495.0000","-376,631.0000",-2.1998
3,"attendances, att_t2",non-booked only,"604,486.0000","611,420.0000","-6,934.0000",-1.1341
4,"attendances, att_other",non-booked only,"9,620,243.0000","10,243,110.0000","-622,867.0000",-6.0808
5,"attendances, all types",non-booked + booked,"27,976,025.0000","27,976,025.0000",0.0000,0.0000
6,four-hour %,non-booked + booked,74.9288,74.9288,0.0000,0.0000
7,"attendances, att_t1",non-booked + booked,"17,121,495.0000","17,121,495.0000",0.0000,0.0000
8,"attendances, att_t2",non-booked + booked,"611,420.0000","611,420.0000",0.0000,0.0000
9,"attendances, att_other",non-booked + booked,"10,243,110.0000","10,243,110.0000",0.0000,0.0000


In [3]:
# Provider-level check: Table 6 is Type 1 only, so I compare it with my Type 1 column, with and without booked.
t1_by_org = m.groupby('org_code')[['att_t1', 'bkd_t1']].sum()
j = t6.set_index('org_code').join(t1_by_org, how='left')
j['resid_non_booked'] = j.attendances - j.att_t1
j['resid_with_booked'] = j.attendances - (j.att_t1 + j.bkd_t1)
prov = pd.DataFrame({'definition': ['non-booked only', 'non-booked + booked'],
                     'providers': len(j),
                     'exact_matches': [(j.resid_non_booked == 0).sum(), (j.resid_with_booked == 0).sum()],
                     'sum_abs_residual': [j.resid_non_booked.abs().sum(), j.resid_with_booked.abs().sum()]})
prov

,definition,providers,exact_matches,sum_abs_residual
0,non-booked only,121,60,376631
1,non-booked + booked,121,121,0


In [4]:
# I write the residuals out whether or not the gate passes, then decide.
DECLARED = 'non-booked + booked'
gate.to_parquet(c.GOLD / 'gate_residuals.parquet', index=False)
prov.to_parquet(c.GOLD / 'gate_provider_residuals.parquet', index=False)
declared = gate[gate.definition == DECLARED]
print('residuals under the declared definition:'); print(declared[['metric', 'residual']].to_string(index=False))
assert (declared.residual.abs() < 1e-6).all(), 'The gate failed: I should not derive anything new until this is explained.'
assert prov.loc[prov.definition == DECLARED, 'exact_matches'].iloc[0] == len(j)
print('Gate passed under', DECLARED)

residuals under the declared definition:
                metric  residual
attendances, all types    0.0000
           four-hour %    0.0000
   attendances, att_t1    0.0000
   attendances, att_t2    0.0000
attendances, att_other    0.0000
Gate passed under non-booked + booked
